In [1]:
!rm .fleche -rf

# Getting Started with Fleche

This notebook demonstrates the main features of the `fleche` library, a caching library for Python.

## Long-running calculation

In [2]:
import time
from fleche import fleche, cache, tags, project
from fleche.digest import Digest

In [3]:
@fleche
def long_running_calculation(x):
    print(f'Running calculation for {x}...')
    time.sleep(2)
    return x * x

In [4]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'First call took {end - start:.2f} seconds.')

Running calculation for 2...
First call took 2.00 seconds.


In [5]:
start = time.time()
long_running_calculation(2)
end = time.time()
print(f'Second call took {end - start:.2f} seconds.')

Second call took 0.00 seconds.


As you can see, the second call returns almost instantly, because the result was cached.

## Recursive function

In [6]:
@fleche
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

In [7]:
start = time.time()
fib(20)
end = time.time()
print(f'fib(20) took {end - start:.4f} seconds with caching.')

fib(20) took 0.0517 seconds with caching.


Without caching, this would be much slower as each call to `fib` would be recomputed.

## Caching Methods of User-defined Types

`fleche` can also cache methods of classes. For this to work, the class must be "digest-compatible". You can make a class digest-compatible by implementing a `__digest__` method or by using a `dataclass`.

In [ ]:
class MyClass:
    def __init__(self, val):
        self.val = val
    
    def __digest__(self):
        # The digest defines how the instance is identified in the cache
        return Digest(str(self.val))

    @fleche
    def compute(self, x):
        print(f"Computing {self.val} + {x}...")
        time.sleep(1)
        return self.val + x

In [ ]:
obj = MyClass(10)

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"First call took {time.time() - start:.2f} seconds.")

start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Second call (same instance) took {time.time() - start:.2f} seconds.")

If you mutate the instance such that its digest changes, the cache will be missed.

In [ ]:
obj.val = 20
start = time.time()
print(f"Result: {obj.compute(5)}")
print(f"Call after mutation took {time.time() - start:.2f} seconds.")

## Passing Digests as Arguments

`fleche` supports passing `Digest` objects directly to cached functions. When a function receives a `Digest`, `fleche` automatically expands it to its actual value from the cache before executing the function. You can use the convenience wrapper `D` to mark a string as a digest.

In [ ]:
from fleche import D

@fleche
def double(x):
    print(f"Doubling {x}...")
    return x * 2

# 1. Calculate long_running_calculation(10) and get its digest
print(long_running_calculation.digest(10))
# Output: e167a117b3ad0edf6646876105f96307185bc0394628d63304a0efec1027abac

# 2. Pass the digest (even a short one!) to double(). It will be expanded to 100.
# Use D() to mark it as a digest.
print(f"Result: {double(D('e167a117'))}")

## Metadata

`fleche` allows you to add metadata to your cached functions using the `tags` context manager. This can be useful for organizing and querying your results.

In [8]:
@fleche
def another_calculation(a, b):
    return a + b

In [9]:
with tags(project='my_project', category='testing'):
    another_calculation(1, 2)
    another_calculation(3, 4)

This metadata is stored alongside the cached result. You can then use the `metadata.table` method to view the metadata for all cached results.

In [10]:
cache().metadata.table()

## Filtering

The metadata table is just pandas so you can query and filter as you like.

In [13]:
cache().metadata.table().query('name!="fib"')

In [14]:
cache().metadata.table().query('project=="my_project"')